# 7) Voltage and reactive power

## Objectives

At the end of this notebook you should be able to:

- explain why a transmission line both **produces and consumes** reactive power, and compute the load at which the two balance — the **surge impedance loading** (SIL)
- show with the AC power flow why a lightly loaded high-voltage grid suffers from **high** voltages, not low ones
- measure how much a bus's voltage moves per Mvar and per MW, and why **reactive power is a local quantity**
- size a **shunt reactor**, and see why its location — and its ability to vary — matters

Lessons 3 to 6 used the DC approximation, which fixes every voltage at 1 p.u. and ignores reactive power. Lesson 3 already found an overload it could not see. This lesson goes back to the full AC power flow and makes voltage and reactive power the subject — on the same stylised Croatian 400 kV network, where they are not an academic problem.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)

### The network in AC

Lesson 3's network and line data (400 kV, 0.028 / 0.32 Ω/km, 11.5 nF/km), lesson 4's plants. Two operating points:

- **peak**: full load (1,900 MW, reactive load 20 % of active), the gas, hydro and coal plants online and holding their voltage at 1.0 p.u., wind at 600 MW;
- **night**: 35 % of the load, no plants online and no wind — everything comes through the grid connection at Zagreb, which holds 1.0 p.u.

In [2]:
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
R_KM, X_KM, C_KM = 0.028, 0.32, 11.5                      # ohm/km, ohm/km, nF/km
LOAD = {"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250}
PLANTS = {"Rijeka": 500, "Split": 600, "Osijek": 300}       # MW when online, voltage set point 1.0 p.u.
V_MAX, V_MIN = 1.05, 0.95


def network(load_scale=1.0, plants=tuple(PLANTS), wind=600.0):
    """The stylised Croatian 400 kV network in pandapower."""
    net = pp.create_empty_network(sn_mva=100)
    bus = {name: pp.create_bus(net, 400, name=name) for name in BUSES}
    pp.create_ext_grid(net, bus["Zagreb"], vm_pu=1.0)
    for name in plants:
        pp.create_gen(net, bus[name], p_mw=PLANTS[name], vm_pu=1.0, name=f"plant {name}")
    pp.create_sgen(net, bus["Velebit"], p_mw=wind, q_mvar=0.0, name="wind Velebit")
    for name, p in LOAD.items():
        pp.create_load(net, bus[name], p_mw=p * load_scale, q_mvar=0.2 * p * load_scale, name=f"load {name}")
    for _, line in LINES.iterrows():
        pp.create_line_from_parameters(net, bus[line.bus0], bus[line.bus1], line.km, R_KM, X_KM, C_KM,
                                       max_i_ka=2.0, name=f"{line.bus0}-{line.bus1}")
    return net, bus


def peak():
    return network()


def night():
    return network(load_scale=0.35, plants=(), wind=0.0)


B_KM = 2 * np.pi * 50 * C_KM * 1e-9                        # shunt susceptance, S/km
print(f"line charging at 400 kV: {400 ** 2 * B_KM:.3f} Mvar per km, {400 ** 2 * B_KM * LINES.km.sum():.0f} Mvar "
      f"for all {LINES.km.sum()} km")

line charging at 400 kV: 0.578 Mvar per km, 751 Mvar for all 1300 km


### A line produces and consumes reactive power

A line's shunt capacitance produces reactive power, V² · B per km — independent of the load. Its series reactance consumes I² · X per km — growing with the square of the load. The two balance at a particular power, the **surge impedance loading**:

V² B = I² X = (P / V)² X  ⟹  **SIL = V² / Z_c**, with the surge impedance **Z_c = √(X / B)**

Below SIL a line is a net **source** of reactive power and lifts the voltage; above SIL it is a net **sink**. The cell checks this on a single 200 km line (without resistance, to isolate the effect) with pandapower's AC power flow.

In [3]:
Zc = np.sqrt(X_KM / B_KM)
SIL = 400 ** 2 / Zc
print(f"surge impedance {Zc:.1f} ohm, SIL {SIL:.1f} MW")


def line_reactive_balance(p_mw, km=200):
    """Net reactive power consumed by a lossless line carrying p_mw between two buses held at 1.0 p.u."""
    net = pp.create_empty_network()
    a, b = pp.create_bus(net, 400), pp.create_bus(net, 400)
    pp.create_ext_grid(net, a, vm_pu=1.0)
    pp.create_gen(net, b, p_mw=0.0, vm_pu=1.0)
    pp.create_load(net, b, p_mw=p_mw)
    pp.create_line_from_parameters(net, a, b, km, 0.0, X_KM, C_KM, max_i_ka=5.0)
    pp.runpp(net, numba=False)
    return net.res_line.ql_mvar.iloc[0]


p = np.linspace(0, 800, 33)
q = np.array([line_reactive_balance(x) for x in p])
crossing = np.interp(0, q, p)
print(f"pandapower: the 200 km line's net reactive balance crosses zero at {crossing:.0f} MW")
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(p, q, color="tab:blue")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(SIL, color="tab:red", ls="--", lw=1, label=f"SIL = {SIL:.0f} MW")
ax.fill_between(p, q, 0, where=q < 0, color="tab:orange", alpha=0.2, label="line produces Mvar: voltage rises")
ax.set(xlabel="active power carried (MW)", ylabel="net reactive power consumed (Mvar)",
       title="One 200 km, 400 kV line")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

surge impedance 297.6 ohm, SIL 537.6 MW


pandapower: the 200 km line's net reactive balance crosses zero at 534 MW


The surge impedance of these lines is 298 Ω, so their SIL is **538 MW**; pandapower's AC power flow puts the zero crossing of the 200 km line at 534 MW — close, the small difference coming from the single π-section line model and the coarse sweep. Unloaded, the line produces 116 Mvar (0.578 Mvar per km × 200 km); at 800 MW it consumes 147 Mvar. A 400 kV line spends most hours well below its SIL, and so most hours it **adds** reactive power to the grid. That is the opposite of the picture most people carry from low-voltage grids, where loads pull voltages down: on a lightly loaded high-voltage grid, the network itself pushes them up.

### Peak and night

In [4]:
def solve(net):
    pp.runpp(net, numba=False)
    return net


results = {}
for label, make in {"peak": peak, "night": night}.items():
    net = solve(make()[0])
    results[label] = net
    flows = net.res_line.p_from_mw.abs()
    print(f"{label:5s}: largest line flow {flows.max():4.0f} MW (SIL {SIL:.0f}); lines produce "
          f"{-net.res_line.ql_mvar.sum():4.0f} Mvar net; grid connection Q {net.res_ext_grid.q_mvar.iloc[0]:5.0f} Mvar; "
          f"plants Q {net.res_gen.q_mvar.round(0).tolist()}")
voltages = pd.DataFrame({label: net.res_bus.vm_pu.to_numpy() for label, net in results.items()}, index=BUSES)
voltages.round(3)

peak : largest line flow  568 MW (SIL 538); lines produce  557 Mvar net; grid connection Q   135 Mvar; plants Q [-33.0, -169.0, -110.0]


night: largest line flow  150 MW (SIL 538); lines produce  782 Mvar net; grid connection Q  -649 Mvar; plants Q []


          peak  night
Zagreb   1.000  1.000
Rijeka   1.000  1.046
Split    1.000  1.093
Osijek   1.000  1.084
Velebit  1.014  1.076

At **peak** the plants hold every voltage at 1.0 p.u. (only Velebit, where the wind farm runs at zero reactive power, rises to 1.014). But look at what they do to hold it: **all three absorb reactive power** — 33, 169 and 110 Mvar — even at full load, because the lines together still produce 557 Mvar net. Only one line, carrying 568 MW, is just above its SIL.

At **night** the plants are offline, the largest flow is 150 MW — a quarter of SIL — and the lines produce 782 Mvar that nobody nearby consumes. The grid connection at Zagreb absorbs 649 Mvar and holds Zagreb at 1.0 p.u., but the far ends of the network rise: **Split 1.093, Osijek 1.084 and Velebit 1.076 p.u.**, above the usual 1.05 p.u. limit for 400 kV. Split is the worst because it sits at the end of the two longest lines (320 and 330 km) and furthest from the voltage control at Zagreb. High voltages age insulation and can trip overvoltage protection — a real operating problem, not a curiosity.

### Reactive power is local

How much does a bus's voltage move when 10 Mvar — or 10 MW — is injected somewhere? The table gives the change in thousandths of a p.u. per Mvar (or per MW), at night, for an injection at each bus (rows) on the voltage of each bus (columns).

In [5]:
base = solve(night()[0]).res_bus.vm_pu.to_numpy()
rows = {}
for source in BUSES[1:]:
    for kind in ("Mvar", "MW"):
        net, bus = night()
        pp.create_sgen(net, bus[source], p_mw=10.0 if kind == "MW" else 0.0, q_mvar=10.0 if kind == "Mvar" else 0.0)
        rows[(source, kind)] = (solve(net).res_bus.vm_pu.to_numpy() - base) / 10 * 1000
sensitivity = pd.DataFrame(rows, index=BUSES).T
sensitivity.index.names = ["injection at", "per"]
sensitivity.round(3)

                   Zagreb  Rijeka  Split  Osijek  Velebit
injection at per                                         
Rijeka       Mvar     0.0   0.203  0.096   0.044    0.161
             MW       0.0   0.024  0.013   0.007    0.019
Split        Mvar     0.0   0.092  0.274   0.124    0.167
             MW       0.0   0.017  0.038   0.022    0.027
Osijek       Mvar     0.0   0.042  0.126   0.333    0.077
             MW       0.0   0.008  0.019   0.042    0.013
Velebit      Mvar     0.0   0.156  0.170   0.078    0.286
             MW       0.0   0.022  0.023   0.013    0.033

Two patterns stand out. First, at each bus a Mvar moves the voltage **7 to 9 times more than a MW** (at Split 0.274 against 0.038 thousandths of a p.u.). That is lesson 2's decoupling seen from the other side: on lines whose reactance dwarfs their resistance, active power mainly moves the voltage **angles** and reactive power the voltage **magnitudes**. Second, the effect is **local**: a Mvar at Osijek moves Osijek's voltage by 0.333, Rijeka's by only 0.042 — eight times less. Reactive power sent across the network is consumed along the way in the lines' reactance, and its effect fades with every kilometre. That is why voltage is controlled where the problem is, with local devices — and why the location of a reactor matters as much as its size.

### Sizing a shunt reactor

A shunt reactor absorbs reactive power, V² / X_reactor — the opposite of the line charging. How large must it be to bring every voltage at night down to 1.05 p.u., and where should it go? Bisection on the reactor size, for each bus in turn:

In [6]:
def max_voltage_with_reactor(bus_name, mvar, make=night):
    net, bus = make()
    pp.create_shunt(net, bus[bus_name], q_mvar=mvar, p_mw=0.0)    # positive q_mvar: absorbs, i.e. a reactor
    return solve(net).res_bus.vm_pu.max()


def smallest_reactor(bus_name, limit=V_MAX, high=2000.0, tol=1.0):
    low = 0.0
    while high - low > tol:
        mid = (low + high) / 2
        low, high = (mid, high) if max_voltage_with_reactor(bus_name, mid) > limit else (low, mid)
    return high


reactors = pd.Series({name: smallest_reactor(name) for name in BUSES[1:]}, name="smallest reactor (Mvar)")
reactors.round(0).to_frame()

         smallest reactor (Mvar)
Rijeka                     834.0
Split                      246.0
Osijek                     321.0
Velebit                    429.0

In [7]:
best = reactors.idxmin()
net, bus = night()
pp.create_shunt(net, bus[best], q_mvar=reactors[best], p_mw=0.0)
voltages[f"night + {reactors[best]:.0f} Mvar at {best}"] = solve(net).res_bus.vm_pu.to_numpy()
fig, ax = plt.subplots(figsize=(8, 3.4))
voltages.plot.bar(ax=ax, rot=0, color=["tab:green", "tab:red", "tab:purple"], width=0.75)
ax.axhline(V_MAX, color="black", ls="--", lw=1)
ax.axhline(1.0, color="grey", lw=0.6)
ax.set(ylim=(0.95, 1.12), ylabel="voltage (p.u.)", title="Voltages at peak, at night, and at night with a reactor")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()
voltages.round(3)

          peak  night  night + 246 Mvar at Split
Zagreb   1.000  1.000                      1.000
Rijeka   1.000  1.046                      1.021
Split    1.000  1.093                      1.018
Osijek   1.000  1.084                      1.050
Velebit  1.014  1.076                      1.030

A reactor at **Split** needs **246 Mvar**; the same job from Rijeka needs **834 Mvar**, more than three times as much. Split has the highest voltage, and a reactor there also pulls down its neighbours Osijek and Velebit strongly (the sensitivity table's Split row). A reactor at Rijeka sits close to Zagreb's strong voltage control and reaches Split only weakly. With 246 Mvar at Split every voltage is within the limit — Osijek exactly at 1.05 p.u., which is what makes this the smallest sufficient size.

This is a real problem in this part of Europe. In the SINCRO.GRID project, completed in 2022, the Croatian and Slovenian transmission system operators installed compensation against overvoltages in lightly loaded periods; in Croatia, HOPS installed a 200 Mvar variable shunt reactor at Melina near Rijeka, a 100 Mvar variable shunt reactor at Mraclin, and a static var compensator (−250 to +70 Mvar) at Konjsko near Split, all at 220 kV. The stylised network here is not the real grid, so its best location is no verdict on theirs — but note the word **variable**: the need changes from hour to hour, which the exercise explores.

## Check your understanding

**Reactive power and voltage**

1. At peak only one line carries slightly more than its SIL, and the lines together still produce 557 Mvar. What does that mean for the plants' reactive power at peak, and for the voltages when the plants are offline?
2. A Mvar injected at Osijek moves Osijek's voltage eight times more than Rijeka's. Why can reactive power not be "transported" across the network the way active power can?

**Compensation**

1. Why does a reactor at Split need less than a third of the Mvar a reactor at Rijeka needs?
2. The reactor sized here fixes one night at 35 % load. What could go wrong with a fixed reactor of that size on a lighter night, or on a busy day?